# PRISM Dataset EDA

**Goal:** Understand the data before writing any training code.

**Run on:** EC2 instance with 3 sessions synced from S3 (~60-100 GB).  
**Do NOT run on full 1.6 TB** — a sample of 3 sessions is enough.

Questions this notebook answers:
1. What do RGB and polar images actually look like side by side?
2. Are pixel values in [0, 4095] (12-bit) or [0, 65535] (16-bit padded)?
3. What does DoLP look like as a heatmap on dry vs wet road?
4. How many frames per surface state / weather condition?
5. What are the per-channel statistics for polar images? (needed for normalization)
6. Is there obvious class imbalance?

In [ ]:
# ── CONFIG — edit these before running ──────────────────────────────────────
LOCAL_DATA_DIR = "/mnt/nvme/polaris"          # where sessions are synced
SAMPLE_SESSIONS = None                         # None = use whatever is in LOCAL_DATA_DIR
# ────────────────────────────────────────────────────────────────────────────

import os, sys
sys.path.insert(0, os.path.abspath(".."))

from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd
from tqdm import tqdm

from src.data.stokes import compute_stokes

plt.rcParams['figure.dpi'] = 120
DATA_ROOT = Path(LOCAL_DATA_DIR)

sessions = SAMPLE_SESSIONS or sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"Sessions found: {sessions}")

## 1. Directory Structure — verify the layout

In [ ]:
session_dir = DATA_ROOT / sessions[0]
print(f"Session: {sessions[0]}")
for p in sorted(session_dir.rglob("*"))[:40]:
    print(' ', p.relative_to(session_dir))

## 2. Pixel Value Range — is this 12-bit or 16-bit full range?

In [ ]:
# Load one frame from each angle and check min/max
session_dir = DATA_ROOT / sessions[0]
polar_0_dir = session_dir / "polar" / "0"
sample_frame = sorted(polar_0_dir.glob("*.tiff"))[0]

for angle in ["0", "45", "90", "135"]:
    path = session_dir / "polar" / angle / sample_frame.name
    arr = tifffile.imread(str(path))
    print(f"  Angle {angle:>3}° | dtype={arr.dtype} | min={arr.min():6d} | max={arr.max():6d} | shape={arr.shape}")

# Also check RGB
rgb_dir = session_dir / "rgb"
rgb_frame = sorted(rgb_dir.glob("*.tiff"))[0]
rgb_arr = tifffile.imread(str(rgb_frame))
print(f"  RGB            | dtype={rgb_arr.dtype} | min={rgb_arr.min():6d} | max={rgb_arr.max():6d} | shape={rgb_arr.shape}")

print()
print("EXPECTED: max ~4095 = 12-bit packed into 16-bit. If max ~65535, it's 16-bit full range.")

## 3. Visual: RGB vs All 4 Polar Angles — same frame

In [ ]:
def load_and_norm(path, bit_depth=12):
    """Load TIFF and normalise to [0,1] float32 for display."""
    arr = tifffile.imread(str(path)).astype(np.float32)
    max_val = (2 ** bit_depth) - 1
    return np.clip(arr / max_val, 0, 1)

session_dir = DATA_ROOT / sessions[0]
frame_stem = sorted((session_dir / "polar" / "0").glob("*.tiff"))[10].stem  # pick frame #10

rgb = load_and_norm(session_dir / "rgb" / f"{frame_stem}.tiff")
p0  = load_and_norm(session_dir / "polar" / "0"   / f"{frame_stem}.tiff")
p45 = load_and_norm(session_dir / "polar" / "45"  / f"{frame_stem}.tiff")
p90 = load_and_norm(session_dir / "polar" / "90"  / f"{frame_stem}.tiff")
p135= load_and_norm(session_dir / "polar" / "135" / f"{frame_stem}.tiff")

# Downsample for display (images are 2448x2048)
scale = 4
def ds(img): return img[::scale, ::scale] if img.ndim == 2 else img[::scale, ::scale, :]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
titles = ["RGB", "Polar 0°", "Polar 45°", "Polar 90°", "Polar 135°", "(empty)"]
imgs   = [rgb, p0, p45, p90, p135, None]

for ax, title, img in zip(axes.flat, titles, imgs):
    if img is None:
        ax.axis("off")
        continue
    if img.ndim == 2:
        ax.imshow(ds(img), cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(ds(img))
    ax.set_title(title, fontsize=13)
    ax.axis("off")

plt.suptitle(f"Frame: {frame_stem}  |  Session: {sessions[0]}", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Stokes Parameters and DoLP Heatmap

In [ ]:
def load_raw(path):
    return tifffile.imread(str(path)).astype(np.float32)

i0   = load_raw(session_dir / "polar" / "0"   / f"{frame_stem}.tiff")
i45  = load_raw(session_dir / "polar" / "45"  / f"{frame_stem}.tiff")
i90  = load_raw(session_dir / "polar" / "90"  / f"{frame_stem}.tiff")
i135 = load_raw(session_dir / "polar" / "135" / f"{frame_stem}.tiff")

stokes = compute_stokes(i0, i45, i90, i135, eps=1.0)

print("Stokes statistics:")
for name, arr in stokes.items():
    print(f"  {name:5s} | min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}  std={arr.std():.4f}")

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
cmaps = {"S0": "gray", "S1": "RdBu_r", "S2": "RdBu_r", "DoLP": "hot", "AoLP": "hsv"}

for ax, (name, arr) in zip(axes, stokes.items()):
    im = ax.imshow(arr[::scale, ::scale], cmap=cmaps[name])
    ax.set_title(name, fontsize=13)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Stokes Parameters — DoLP shows polarization degree (high = wet/icy)", fontsize=11)
plt.tight_layout()
plt.show()

## 5. DoLP Comparison: Dry vs Wet Road (if multiple sessions available)

In [ ]:
# Replace session names below with actual dry/wet sessions from your data
# Run cell 1 first to see what sessions are available

CONDITION_SESSIONS = {
    "dry":  sessions[0] if len(sessions) > 0 else None,
    "wet":  sessions[1] if len(sessions) > 1 else None,
    "snow": sessions[2] if len(sessions) > 2 else None,
}

fig, axes = plt.subplots(len(CONDITION_SESSIONS), 3, figsize=(16, 5 * len(CONDITION_SESSIONS)))
if len(CONDITION_SESSIONS) == 1:
    axes = [axes]

for row, (condition, session_name) in enumerate(CONDITION_SESSIONS.items()):
    if session_name is None:
        for ax in axes[row]: ax.axis("off")
        continue

    sd = DATA_ROOT / session_name
    frame = sorted((sd / "polar" / "0").glob("*.tiff"))[5].stem

    rgb_img = load_and_norm(sd / "rgb" / f"{frame}.tiff")
    i0_   = load_raw(sd / "polar" / "0"   / f"{frame}.tiff")
    i45_  = load_raw(sd / "polar" / "45"  / f"{frame}.tiff")
    i90_  = load_raw(sd / "polar" / "90"  / f"{frame}.tiff")
    i135_ = load_raw(sd / "polar" / "135" / f"{frame}.tiff")
    st    = compute_stokes(i0_, i45_, i90_, i135_)

    axes[row][0].imshow(ds(rgb_img)); axes[row][0].set_title(f"{condition.upper()} — RGB"); axes[row][0].axis("off")
    im1 = axes[row][1].imshow(st["DoLP"][::scale, ::scale], cmap="hot", vmin=0, vmax=0.5)
    axes[row][1].set_title("DoLP"); axes[row][1].axis("off")
    plt.colorbar(im1, ax=axes[row][1])
    im2 = axes[row][2].imshow(st["AoLP"][::scale, ::scale], cmap="hsv")
    axes[row][2].set_title("AoLP"); axes[row][2].axis("off")
    plt.colorbar(im2, ax=axes[row][2])

plt.suptitle("DoLP per surface condition — wet/icy should show higher DoLP on road region", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Label / Metadata Structure — inspect what labels look like

In [ ]:
# Inspect the session directory for label files (JSON, CSV, YAML)
# The format is unknown until we see the actual data
session_dir = DATA_ROOT / sessions[0]
label_files = list(session_dir.glob("*.json")) + list(session_dir.glob("*.csv")) + list(session_dir.glob("*.yaml"))
print(f"Label files found: {label_files}")

if label_files:
    import json
    with open(label_files[0]) as f:
        sample = json.load(f) if label_files[0].suffix == '.json' else f.read()
    print("\nFirst label file sample:")
    print(str(sample)[:2000])
else:
    print("No label files found at session root — check dataset card for label format")

## 7. Class Distribution (fill in after label format is confirmed)

In [ ]:
# TODO: populate df_labels after understanding label file format from cell above
# Example of what this should produce:
#   df_labels = pd.DataFrame({'surface': [...], 'weather': [...], 'material': [...]})

# Placeholder — replace with real label loading
print("Fill this cell after confirming label format in cell 6.")

# Once df_labels is ready, run:
# fig, axes = plt.subplots(1, 3, figsize=(18, 4))
# for ax, col in zip(axes, ['surface', 'weather', 'material']):
#     df_labels[col].value_counts().plot.bar(ax=ax, title=col)
# plt.tight_layout(); plt.show()

## 8. Per-Channel Statistics for Polar Images (needed for normalization)

In [ ]:
# Sample 50 random frames from each available session and compute Stokes stats
# These mean/std values go into configs/transforms.py

import random

channel_names = ["S0", "S1", "S2", "DoLP", "AoLP"]
all_means = {c: [] for c in channel_names}
all_stds  = {c: [] for c in channel_names}

SAMPLE_N = 50  # frames per session

for session_name in sessions:
    sd = DATA_ROOT / session_name
    frames = sorted((sd / "polar" / "0").glob("*.tiff"))
    sampled = random.sample(frames, min(SAMPLE_N, len(frames)))

    for f in tqdm(sampled, desc=session_name, leave=False):
        stem = f.stem
        try:
            i0_   = load_raw(sd / "polar" / "0"   / f"{stem}.tiff")
            i45_  = load_raw(sd / "polar" / "45"  / f"{stem}.tiff")
            i90_  = load_raw(sd / "polar" / "90"  / f"{stem}.tiff")
            i135_ = load_raw(sd / "polar" / "135" / f"{stem}.tiff")
            st = compute_stokes(i0_, i45_, i90_, i135_)
            for ch in channel_names:
                all_means[ch].append(float(st[ch].mean()))
                all_stds[ch].append(float(st[ch].std()))
        except Exception as e:
            print(f"  skipping {stem}: {e}")

print("\nPolar channel statistics (copy into src/data/transforms.py):")
polar_mean = []
polar_std  = []
for ch in channel_names:
    m = np.mean(all_means[ch])
    s = np.mean(all_stds[ch])
    polar_mean.append(round(float(m), 6))
    polar_std.append(round(float(s), 6))
    print(f"  {ch:5s}  mean={m:.4f}  std={s:.4f}")

print(f"\npolar_mean = {polar_mean}")
print(f"polar_std  = {polar_std}")

## Summary

After running this notebook, update the following before writing training code:

| Finding | Where to update |
|---|---|
| 12-bit max value (4095 vs 65535) | `src/data/dataset.py` normalization |
| Label file format and field names | `src/data/dataset.py` `_index_sessions()` |
| polar_mean / polar_std values | `src/data/transforms.py` |
| Actual session IDs and their conditions | `configs/splits/session_splits.yaml` |
| Class imbalance found | `src/training/trainer.py` → add class weights |